# 🔧 Orqest Tutorial 2: Agents as Tools

Welcome to **Agent as Tools** - one of Orqest's most powerful patterns! This tutorial shows how agents can use other agents as tools.

---

## 🎯 What You'll Learn

- How to use one agent as a tool for another agent
- Why specialized agents are better than one big agent
- How to maintain conversation context between agents

---

## 🌟 The Core Idea: Specialized Agents

Instead of one agent trying to do everything:

| ❌ Traditional | ✅ Orqest Way |
|---------------|---------------|
| One big agent doing planning, chatting, analyzing | Specialized agents: one for planning, one for orchestrating |
| Complex prompts | Simple, focused prompts |
| Hard to debug | Easy to trace which agent did what |

Let's build this! 👇


---

## 📦 Step 1: Imports

Same as Tutorial 1, but we add `RunContext` for tools:


In [72]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any
from pydantic_ai import RunContext

from orqest.agents.base_agent import BaseAgent, NoValidResponse


---

## 🧠 Step 2: Define State and Output Models

Following Orqest's pattern from Tutorial 1, we separate:
- **Global State**: The conversation memory (SimpleState)
- **Output Formats**: Structured response formats (PlanOutput, TextOutput)

This separation gives us the same benefits as Tutorial 1:
- ✅ **Clear Contracts**: Each agent's output format is explicit
- ✅ **Type Safety**: IDEs can autocomplete and catch errors
- ✅ **Easy Integration**: Other components know exactly what to expect
- ✅ **Validation**: Pydantic ensures outputs meet requirements


In [73]:
class SimpleState(BaseModel):
    """Simple state with just messages and a plan."""

    messages: List[Dict[str, str]] = Field(default_factory=list)
    plan: List[str] = Field(default_factory=list)

    def add_message(self, role: str, content: str) -> None:
        """Add a message to the conversation."""
        self.messages.append({"role": role, "content": content})

    def get_user_message(self) -> str:
        """Get the latest user message."""
        for msg in reversed(self.messages):
            if msg["role"] == "user":
                return msg["content"]
        return "No user message"

class PlanOutput(BaseModel):
    """Structured output format for planning responses.

    This defines what the planner agent returns to pydantic-ai,
    separate from the conversation state.
    """
    plan_text: str = Field(
        description="The planning response text",
        min_length=1
    )
    steps_identified: int = Field(
        description="Number of plan steps identified",
        ge=0
    )

class TextOutput(BaseModel):
    """Simple text output format for general responses.

    Used by the orchestrator for non-planning responses.
    """
    answer: str = Field(
        description="The assistant's reply text",
        min_length=1
    )


---

## 🎯 Step 3: Planning Agent (The Specialist)

This agent does ONE thing: creates plans. That's it!


In [74]:
class PlannerAgent(BaseAgent[SimpleState]):
    """A simple planning agent that creates step-by-step plans."""

    def __init__(self):
        super().__init__(
            agent_name="planner",
            output_type=PlanOutput,  # Uses PlanOutput for pydantic-ai
            system_prompt="You are a planning agent. Break down the user's request into 3-5 clear, numbered steps. Be practical and specific.",
            retries=2,
            deps_type=SimpleState
        )

    async def _run_implementation(self, state: SimpleState, **kwargs) -> SimpleState:
        """Create a plan based on the user's request."""
        user_request = state.get_user_message()

        prompt = f"Create a step-by-step plan for: {user_request}"

        response = await self.agent.run(prompt, deps=state, **kwargs)

        return await self._process_response_implementation(response, state, **kwargs)

    async def _process_response_implementation(self, response, state: SimpleState, **kwargs) -> SimpleState:
        """Extract the plan from the response."""
        # Extract content from PlanOutput response
        if hasattr(response, "output") and hasattr(response.output, "plan_text"):
            content = response.output.plan_text
        elif hasattr(response, "content"):
            content = response.content
        else:
            content = str(response)

        # Add response to messages
        state.add_message("assistant", content)

        # Extract numbered steps (simple parsing)
        state.plan.clear()
        lines = content.split('\n')
        for line in lines:
            line = line.strip()
            if line and (line[0].isdigit() or line.startswith(("Step", "•", "-"))):
                # Clean up the step
                clean_step = line.split('.', 1)[-1].strip()
                if clean_step:
                    state.plan.append(clean_step)

        return state


---

## 🎭 Step 4: Orchestrator Agent (Uses Other Agents as Tools)

This is where the magic happens! The orchestrator uses the planner as a **tool**.


In [75]:
class OrchestratorAgent(BaseAgent[SimpleState]):
    """Orchestrator that uses other agents as tools."""

    def __init__(self):
        # Set up tools - including other agents!
        tools = [self._use_planner]

        super().__init__(
            agent_name="orchestrator",
            output_type=TextOutput,  # Uses TextOutput for pydantic-ai
            system_prompt="""You are a helpful orchestrator. When users ask for help planning something, use the 'use_planner' tool. For other questions, respond directly.""",
            retries=2,
            deps_type=SimpleState,
            tools=tools
        )

        # Create the planner agent
        self.planner = PlannerAgent()

    async def _run_implementation(self, state: SimpleState, **kwargs) -> SimpleState:
        """Analyze the request and decide whether to use tools."""
        user_message = state.get_user_message()

        prompt = f"""
        User said: {user_message}

        If this looks like a planning request (organizing, creating steps, etc.), use the use_planner tool.
        Otherwise, respond directly.
        """

        response = await self.agent.run(prompt, deps=state, **kwargs)

        return await self._process_response_implementation(response, state, **kwargs)

    async def _process_response_implementation(self, response, state: SimpleState, **kwargs) -> SimpleState:
        """Process the orchestrator's response."""
        # Extract content from TextOutput response
        if hasattr(response, "output") and hasattr(response.output, "answer"):
            content = response.output.answer
        elif hasattr(response, "content"):
            content = response.content
        else:
            content = str(response)

        state.add_message("assistant", content)
        return state

    # === THE MAGIC: AGENT AS TOOL ===

    async def _use_planner(self, ctx: RunContext[SimpleState], task: str) -> Dict[str, Any]:
        """🔧 Tool: Use the planner agent to create a plan.

        This is the heart of 'Agent as Tools'!
        """
        try:
            # Get current state
            state = ctx.deps

            # Add the planning task as a user message
            state.add_message("user", task)

            # 🔥 Call the planner agent!
            result = await self.planner.run(state)

            if isinstance(result, NoValidResponse):
                return {"success": False, "error": "Planning failed"}

            return {
                "success": True,
                "plan_steps": len(result.plan),
                "plan": result.plan
            }

        except Exception as e:
            return {"success": False, "error": str(e)}


---

## 🎬 Step 5: See It Work!

Let's test our agent-as-tools system:


In [76]:
# Create initial state
state = SimpleState()
state.add_message("user", "Help me plan a birthday party for my friend")

# Create orchestrator (which has planner as a tool)
orchestrator = OrchestratorAgent()

print("🎭 Running orchestrator...")
result = await orchestrator.run(state)

print("\n✨ Done! Let's see what happened:\n")
print(f"📝 Messages: {len(result.messages)}")
print(f"📋 Plan steps: {len(result.plan)}")

# Show the conversation
for i, msg in enumerate(result.messages):
    role = msg["role"].upper()
    content = msg["content"][:100] + "..." if len(msg["content"]) > 100 else msg["content"]
    print(f"\n{i+1}. [{role}]: {content}")

# Show the plan
if result.plan:
    print(f"\n📋 Generated Plan:")
    for i, step in enumerate(result.plan, 1):
        print(f"  {i}. {step}")


🎭 Running orchestrator...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



✨ Done! Let's see what happened:

📝 Messages: 4
📋 Plan steps: 8

1. [USER]: Help me plan a birthday party for my friend

2. [USER]: Plan a birthday party for a friend

3. [ASSISTANT]: 1. Determine the Date and Time:
   - Discuss with your friend to select a convenient date and time f...

4. [ASSISTANT]: Here's a simple birthday party plan you can follow:

1. **Determine the Date and Time:** Decide when...

📋 Generated Plan:
  1. Determine the Date and Time:
  2. Choose a Venue:
  3. Create a Guest List:
  4. Send Invitations:
  5. Plan the Menu and Activities:
  6. Decorate and Prepare:
  7. Plan for a Cake and Gifts:
  8. Confirm Attendance and Finalize Details:


---

## 🔄 Step 6: Continue the Conversation

The beauty of this approach: the state carries forward naturally!


In [77]:
# Add a follow-up question
result.add_message("user", "Make it more budget-friendly")

print("🎭 Running with follow-up...")
updated_result = await orchestrator.run(result)

print("\n✨ Updated response:")
latest_msg = updated_result.messages[-1]["content"]
print(f"🤖 {latest_msg[:200]}...")

print(f"\n📊 Total messages now: {len(updated_result.messages)}")
print(f"📋 Plan steps now: {len(updated_result.plan)}")


🎭 Running with follow-up...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



✨ Updated response:
🤖 To make your plan more budget-friendly, follow these steps:

1. **Review the Current Plan**: Thoroughly analyze the existing plan to understand all its components, costs, and specifics. Make a detaile...

📊 Total messages now: 8
📋 Plan steps now: 5


---

## 🎯 Step 7: Test Different Request Types

Let's see how the orchestrator handles different types of requests:


In [61]:
# Test 1: Non-planning request
simple_state = SimpleState()
simple_state.add_message("user", "What's the weather like?")

simple_result = await orchestrator.run(simple_state)
print("🧪 Test 1 - Weather question:")
print(f"   Response: {simple_result.messages[-1]['content'][:100]}...")
print(f"   Plan created: {'Yes' if simple_result.plan else 'No'}")

# Test 2: Planning request
planning_state = SimpleState()
planning_state.add_message("user", "Help me plan a study schedule for learning Python")

planning_result = await orchestrator.run(planning_state)
print(f"\n🧪 Test 2 - Study planning:")
print(f"   Response: {planning_result.messages[-1]['content'][:100]}...")
print(f"   Plan steps: {len(planning_result.plan)}")
if planning_result.plan:
    print(f"   First step: {planning_result.plan[0]}")


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


🧪 Test 1 - Weather question:
   Response: I can't provide the current weather information. Please check a reliable weather website or app for ...
   Plan created: No


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



🧪 Test 2 - Study planning:
   Response: Here's a plan to help you learn Python effectively:

1. **Identify Learning Goals:** Determine what ...
   Plan steps: 10
   First step: **Identify Learning Goals:**


---

## 🎓 What You've Built

Congratulations! You've implemented the **Agent as Tools** pattern with:

### 🏗️ **Simple Architecture**
```
🎭 OrchestratorAgent (Coordinator)
└── 🎯 PlannerAgent (Tool/Specialist)
```

### 🌟 **Key Benefits You've Seen**

| Benefit | What You Built |
|---------|----------------|
| **Specialization** | Planner only does planning, orchestrator only coordinates |
| **Reusability** | Planner can be used by other agents too |
| **Clarity** | Easy to see which agent did what |
| **Testability** | Can test each agent independently |

### 🚀 **Why This Scales**

This simple pattern lets you:
- Add more specialist agents (executor, analyzer, etc.)
- Chain agents together for complex workflows
- Mix and match agents for different use cases
- Keep each agent simple and focused

### 🎯 **Next Steps**

Try adding:
1. **ExecutorAgent**: Takes plans and executes them
2. **AnalyzerAgent**: Reviews results and provides insights
3. **More tools**: Web search, database queries, etc.

The pattern stays the same - each agent does one thing well, and agents use each other as tools! 🚀

---

## 🎉 You Did It!

You've mastered the **Agent as Tools** pattern - one of the most powerful concepts in modular AI development.

**The key insight**: Instead of building one complex agent, build simple agents that work together. This makes everything easier to understand, test, and extend!

Welcome to truly modular AI development! 🌟


---

## 📦 Step 1: Import Required Libraries

We'll need a few more imports than Tutorial 1 since we're building a multi-agent system:

```python
# What we're importing:
# 📋 pydantic        → Data validation and type safety
# 🤖 BaseAgent       → Core abstraction for all Orqest agents
# 🔧 typing          → Type hints for better code quality
# 🚫 NoValidResponse → Error handling for failed agent calls
# 🏃 RunContext      → Context management for tool execution
# 📝 logging         → Observability and debugging
```

**💪 Why RunContext is important:** It provides a secure, typed way to pass state between agents and tools!

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional, Type
import logging
from pydantic_ai import RunContext

from orqest.agents.base_agent import BaseAgent, NoValidResponse
from orqest.errors import ErrorSeverity

# Set up logging to see what's happening behind the scenes
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

---

## 🧠 Step 2: Define a Rich State Model

For multi-agent systems, we need a more sophisticated state that can handle:
- **Messages** from multiple agents
- **Plans** created by planning agents
- **Chat History** for context preservation

### 🌟 Why This Design Scales

| Feature | Benefit |
|---------|--------|
| **Separate chat_history** | Maintains full context for LLM interactions |
| **Structured messages** | Easy to filter by role, timestamp, etc. |
| **Plan storage** | Persistent planning across multiple turns |
| **Helper methods** | Clean API for common operations |

### 🎁 Real-World Applications

- **🏢 Enterprise chatbots**: Track user requests across departments
- **🎓 Educational assistants**: Maintain learning progress and plans
- **🛠️ Technical support**: Track issue resolution steps
- **📝 Content creation**: Manage multi-step writing workflows

In [ ]:
class ConversationState(BaseModel):
    """Rich conversation state for multi-agent interactions.
    
    This state model demonstrates how to handle complex interactions
    between multiple agents while maintaining clean separation of concerns.
    """
    
    # Core conversation data
    messages: List[Dict[str, Any]] = Field(
        description="Structured conversation messages with metadata",
        default_factory=list
    )
    
    # Planning and task management
    plan: List[str] = Field(
        description="Current plan steps generated by planning agents",
        default_factory=list
    )
    
    # Chat history for LLM context
    chat_history: List[str] = Field(
        description="Formatted chat history for LLM message_history parameter",
        default_factory=list
    )
    
    # === Message Management Methods ===
    
    def add_message(self, role: str, content: str, agent_name: Optional[str] = None) -> None:
        """Add a message with optional agent tracking.
        
        Args:
            role: 'user', 'assistant', or 'system'
            content: The message content
            agent_name: Which agent generated this message (for debugging)
        """
        message = {
            "role": role,
            "content": content
        }
        
        # Add agent metadata if provided
        if agent_name:
            message["agent_name"] = agent_name
            
        self.messages.append(message)
    
    def get_latest_user_message(self) -> Optional[str]:
        """Get the most recent user message."""
        for message in reversed(self.messages):
            if message["role"] == "user":
                return message["content"]
        return None
    
    def get_latest_assistant_message(self) -> Optional[str]:
        """Get the most recent assistant message."""
        for message in reversed(self.messages):
            if message["role"] == "assistant":
                return message["content"]
        return None
    
    def get_messages_by_agent(self, agent_name: str) -> List[Dict[str, Any]]:
        """Get all messages from a specific agent."""
        return [
            msg for msg in self.messages 
            if msg.get("agent_name") == agent_name
        ]
    
    # === Plan Management Methods ===
    
    def add_plan_step(self, step: str) -> None:
        """Add a step to the current plan."""
        self.plan.append(step)
    
    def clear_plan(self) -> None:
        """Clear the current plan."""
        self.plan.clear()
    
    def get_plan_summary(self) -> str:
        """Get a formatted summary of the current plan."""
        if not self.plan:
            return "No plan created yet."
        
        return "\n".join(f"{i+1}. {step}" for i, step in enumerate(self.plan))
    
    # === Chat History Management ===
    
    def update_chat_history(self, new_messages: List[str]) -> None:
        """Update chat history with new LLM messages.
        
        This is used by agents to maintain context for subsequent LLM calls.
        """
        self.chat_history.extend(new_messages)

---

## 🎯 Step 3: Build the Planning Agent

Our `PlannerAgent` will be a **specialist** that excels at breaking down complex tasks into manageable steps.

### 🧠 Why Specialized Agents Rock

```python
# Instead of one giant prompt:
"You are an AI that can chat, plan, execute, analyze, and do everything..."

# We have focused agents:
"You are a planning agent. Your ONE job is to create detailed plans."
```

**Benefits:**
- 🎯 **Better Performance**: Focused prompts = better results
- 🧪 **Easier Testing**: Test planning logic independently
- 🔄 **Reusability**: Use the same planner across different workflows
- 🛠️ **Maintainability**: Fix planning issues in one place

### 🔧 The Tool Pattern

Notice how we give our agent a **tool** (`analyze_task_complexity`). This demonstrates:
- Tools provide specific capabilities
- Agents can combine LLM reasoning with structured functions
- Each tool has a clear, single responsibility

In [ ]:
class PlannerAgent(BaseAgent[ConversationState]):
    """A specialized planning agent that creates detailed task breakdowns.
    
    This agent demonstrates:
    - Focused responsibility (planning only)
    - Tool integration for enhanced capabilities
    - Proper message history management
    - Robust error handling
    """
    
    def __init__(
        self,
        agent_name: str = "planner_agent",
        system_prompt: Optional[str] = None,
        retries: int = 2
    ):
        """Initialize the planner agent with focused capabilities."""
        
        # Define the agent's specialty
        _system_prompt = system_prompt or self._build_system_prompt()
        
        # Set up tools for enhanced planning
        _tools = [self._analyze_task_complexity]
        
        super().__init__(
            agent_name=agent_name,
            output_type=ConversationState | NoValidResponse,
            system_prompt=_system_prompt,
            retries=retries,
            deps_type=ConversationState,  # This enables RunContext access
            tools=_tools
        )
    
    def _build_system_prompt(self) -> str:
        """Build a focused system prompt for planning tasks."""
        return """
        You are a planning agent specialized in breaking down complex tasks into clear, actionable steps.
        
        Your responsibilities:
        1. Analyze user requests to understand the core objective
        2. Break down complex tasks into 3-7 clear, sequential steps
        3. Consider dependencies between steps
        4. Use the analyze_task_complexity tool when needed
        
        Always provide:
        - A brief summary of what the user wants
        - A numbered list of specific action steps
        - Any important considerations or prerequisites
        
        Keep plans practical and actionable!
        """
    
    async def _run_implementation(self, state: ConversationState, **kwargs) -> ConversationState:
        """🚀 Core planning logic with proper message history management."""
        
        try:
            # Get the user's request
            user_message = state.get_latest_user_message()
            if not user_message:
                # Add a default message if none exists
                state.add_message("user", "Please help me plan a task.", "system")
                user_message = "Please help me plan a task."
            
            # Build the planning prompt
            prompt = f"""
            User's request: {user_message}
            
            Current plan status: {state.get_plan_summary()}
            
            Please analyze this request and create a detailed plan. 
            Use the analyze_task_complexity tool if needed to better understand the task.
            """
            
            # Log the operation for debugging
            logger.info(f"Planning for: {user_message[:100]}...")
            
            # 🔥 KEY FEATURE: Pass message_history for context!
            response = await self.agent.run(
                prompt, 
                deps=state,  # State available to tools via RunContext
                message_history=state.chat_history,  # Maintain conversation context
                **kwargs
            )
            
            # 📝 Update chat history with new messages
            state.update_chat_history(response.all_messages())
            
            # Process the response
            return await self._process_response_implementation(response, state, **kwargs)
            
        except Exception as e:
            logger.error(f"Error in planner agent: {str(e)}")
            
            # Use Orqest's built-in error handling
            return self._handle_agent_error(
                error=e,
                operation="_run_implementation",
                severity=ErrorSeverity.ERROR,
                details={"user_message": user_message}
            )
    
    async def _process_response_implementation(
        self, 
        response, 
        state: ConversationState, 
        **kwargs
    ) -> ConversationState:
        """🔄 Process the agent's response and update the state."""
        
        try:
            # Extract the content from the response
            if hasattr(response, "output") and response.output:
                content = response.content if hasattr(response, "content") else str(response.output)
            else:
                content = str(response)
            
            # Add the planner's response to messages
            state.add_message("assistant", content, self.agent_name)
            
            # Extract and store plan steps (simple parsing for demo)
            # In production, you might want more sophisticated parsing
            if "1." in content or "Step 1" in content:
                # Clear existing plan and extract new steps
                state.clear_plan()
                lines = content.split('\n')
                for line in lines:
                    line = line.strip()
                    # Look for numbered steps
                    if line and (line[0].isdigit() or line.startswith("Step")):
                        # Clean up the step text
                        clean_step = line.split('.', 1)[-1].strip()
                        if clean_step:
                            state.add_plan_step(clean_step)
            
            logger.info(f"Planner generated {len(state.plan)} plan steps")
            return state
            
        except Exception as e:
            logger.error(f"Error processing planner response: {str(e)}")
            
            # Add a friendly error message
            state.add_message(
                "assistant", 
                "I had trouble creating a plan. Could you please rephrase your request?",
                self.agent_name
            )
            
            return state
    
    # === TOOL DEFINITION ===
    
    def _analyze_task_complexity(self, ctx: RunContext[ConversationState], task_description: str) -> Dict[str, str]:
        """🔧 Tool: Analyze the complexity of a given task.
        
        This demonstrates how to create tools that:
        - Access the current state via RunContext
        - Provide structured output
        - Support the agent's decision-making
        
        Args:
            ctx: RunContext containing the current ConversationState
            task_description: Description of the task to analyze
            
        Returns:
            Dict containing complexity analysis
        """
        # Access the current state
        state = ctx.deps
        
        # Simple complexity analysis based on keywords and length
        complexity_indicators = [
            "multiple", "complex", "several", "various", "integrate", 
            "coordinate", "manage", "organize", "analyze", "research"
        ]
        
        task_lower = task_description.lower()
        complexity_score = sum(1 for indicator in complexity_indicators if indicator in task_lower)
        
        # Determine complexity level
        if complexity_score >= 3 or len(task_description) > 200:
            complexity = "high"
            recommendation = "Break into 5-7 detailed steps with clear dependencies"
        elif complexity_score >= 1 or len(task_description) > 100:
            complexity = "medium"
            recommendation = "Create 3-5 clear steps with some detail"
        else:
            complexity = "low"
            recommendation = "Simple 2-3 step plan should suffice"
        
        logger.info(f"Task complexity analysis: {complexity} (score: {complexity_score})")
        
        return {
            "complexity": complexity,
            "score": str(complexity_score),
            "recommendation": recommendation,
            "task_length": str(len(task_description))
        }

---

## 🎭 Step 4: Build the Orchestrator Agent - The Master Controller

The `OrchestratorAgent` is where the **"Agent as Tools"** pattern truly shines. It doesn't do the planning itself - instead, it **delegates** to specialized agents.

### 🌟 The Agent-as-Tools Philosophy

```python
# Traditional approach:
def solve_everything(user_input):
    # 500 lines of mixed logic 😱
    plan = create_plan(user_input)
    result = execute_plan(plan)
    return analyze_result(result)

# Orqest approach:
def orchestrate(user_input):
    plan = planner_agent.run(user_input)      # ✨ Delegate to specialist
    result = executor_agent.run(plan)         # ✨ Each agent has one job
    return analyzer_agent.run(result)         # ✨ Clean, testable, reusable
```

### 🏆 Why This Architecture Wins

| Traditional Monolith | Orqest Agent Composition |
|---------------------|---------------------------|
| **One agent does everything** | **Each agent excels at one thing** |
| Prompt conflicts | Focused, optimized prompts |
| Hard to debug | Trace issues to specific agents |
| All-or-nothing failures | Graceful degradation |
| Impossible to optimize | Optimize each agent independently |
| Rigid workflows | Dynamic, adaptable orchestration |

### 🔄 Message History Magic

Watch how the orchestrator maintains **conversational continuity** while coordinating multiple agents!

In [ ]:
class OrchestratorAgent(BaseAgent[ConversationState]):
    """Master orchestrator that coordinates multiple specialized agents.
    
    This agent demonstrates the "Agent as Tools" pattern:
    - Uses other agents as tools
    - Manages overall conversation flow
    - Maintains message history across agent interactions
    - Provides intelligent delegation and coordination
    """
    
    def __init__(
        self,
        agent_name: str = "orchestrator",
        system_prompt: Optional[str] = None,
        retries: int = 2,
        planner_agent: Optional[PlannerAgent] = None
    ):
        """Initialize the orchestrator with specialized sub-agents."""
        
        # Define the orchestrator's role
        _system_prompt = system_prompt or self._build_system_prompt()
        
        # Set up tools (including other agents!)
        _tools = [self._call_planner_agent]
        
        super().__init__(
            agent_name=agent_name,
            output_type=ConversationState | NoValidResponse,
            system_prompt=_system_prompt,
            retries=retries,
            deps_type=ConversationState,
            tools=_tools
        )
        
        # 🔥 Initialize sub-agents
        self.planner_agent = planner_agent or PlannerAgent()
    
    def _build_system_prompt(self) -> str:
        """Build the orchestrator's system prompt."""
        return """
        You are an orchestrator agent that coordinates multiple specialized agents to help users.
        
        Your role:
        1. Understand what the user needs
        2. Decide which specialized agents can help
        3. Coordinate between agents to provide comprehensive assistance
        4. Synthesize results from different agents into coherent responses
        
        Available tools:
        - call_planner_agent: For breaking down complex tasks into plans
        
        Guidelines:
        - Use specialized agents when their expertise is needed
        - For planning tasks, always use the planner agent
        - Provide clear, helpful responses that synthesize agent outputs
        - Maintain conversational flow and context
        
        Be strategic about when to use tools vs. when to respond directly!
        """
    
    async def _run_implementation(self, state: ConversationState, **kwargs) -> ConversationState:
        """🚀 Core orchestration logic with multi-agent coordination."""
        
        try:
            # Analyze the current state
            user_message = state.get_latest_user_message()
            current_plan = state.get_plan_summary()
            
            # Build context-aware prompt
            prompt = f"""
            Current user message: {user_message}
            Current plan status: {current_plan}
            
            Analyze the situation and determine the best approach:
            - If the user wants help planning something, use the planner agent
            - If they're asking about existing plans, reference the current plan
            - If it's a general question, respond directly
            
            Consider what information is missing and what tools would be most helpful.
            """
            
            logger.info(f"Orchestrating response for: {user_message[:50]}...")
            
            # 🔥 Execute with full message history context
            response = await self.agent.run(
                prompt,
                deps=state,  # Full state available to tools
                message_history=state.chat_history,  # Conversation context
                **kwargs
            )
            
            # 📝 Update message history
            state.update_chat_history(response.all_messages())
            
            # Process the response
            return await self._process_response_implementation(response, state, **kwargs)
            
        except Exception as e:
            logger.error(f"Error in orchestrator: {str(e)}")
            
            return self._handle_agent_error(
                error=e,
                operation="_run_implementation",
                severity=ErrorSeverity.ERROR,
                details={
                    "user_message": user_message,
                    "plan_steps": len(state.plan)
                }
            )
    
    # === AGENT-AS-TOOL IMPLEMENTATION ===
    
    async def _call_planner_agent(self, ctx: RunContext[ConversationState], query: str) -> Dict[str, Any]:
        """🤖 Tool: Use the planner agent to create detailed plans.
        
        This is the heart of the "Agent as Tools" pattern!
        
        Args:
            ctx: RunContext with current conversation state
            query: The planning query to send to the planner
            
        Returns:
            Dict containing the planning results
        """
        try:
            logger.info(f"Calling planner agent with: {query[:50]}...")
            
            # Get current state
            state = ctx.deps
            
            # Add the query as a user message for the planner
            state.add_message("user", query)
            
            # 🔥 Call the specialized planner agent
            result_state = await self.planner_agent.run(state)
            
            # Check for errors
            if isinstance(result_state, NoValidResponse):
                logger.warning(f"Planner agent failed: {result_state.error_message}")
                return {
                    "success": False,
                    "error": result_state.error_message,
                    "plan": []
                }
            
            # Extract results
            plan_summary = result_state.get_plan_summary()
            latest_response = result_state.get_latest_assistant_message()
            
            logger.info(f"Planner generated {len(result_state.plan)} steps")
            
            # Return structured results
            return {
                "success": True,
                "plan": result_state.plan,
                "plan_summary": plan_summary,
                "planner_response": latest_response,
                "total_steps": len(result_state.plan)
            }
            
        except Exception as e:
            logger.error(f"Error calling planner agent: {str(e)}")
            
            return {
                "success": False,
                "error": f"Failed to call planner: {str(e)}",
                "plan": []
            }
    
    async def _process_response_implementation(
        self, 
        response, 
        state: ConversationState, 
        **kwargs
    ) -> ConversationState:
        """🔄 Process orchestrator response and maintain conversation flow."""
        
        try:
            # Extract content
            if hasattr(response, "output") and response.output:
                content = response.content if hasattr(response, "content") else str(response.output)
            else:
                content = str(response)
            
            # Add orchestrator's response
            state.add_message("assistant", content, self.agent_name)
            
            logger.info("Orchestrator response processed successfully")
            return state
            
        except Exception as e:
            logger.error(f"Error processing orchestrator response: {str(e)}")
            
            # Add fallback message
            state.add_message(
                "assistant",
                "I encountered an issue while processing your request. Could you please try again?",
                self.agent_name
            )
            
            return state

---

## 🎬 Step 5: See the Magic in Action - Multi-Agent Orchestration

Now let's see how our **Agent-as-Tools** system works in practice. Watch how:

1. **User** makes a request
2. **Orchestrator** analyzes and decides to use the planner
3. **Planner** creates a detailed plan using its tools
4. **Orchestrator** synthesizes everything into a coherent response
5. **Message History** maintains context throughout

### 🔍 What to Watch For

- **Automatic delegation**: Orchestrator knows when to call the planner
- **Tool usage**: Planner uses `analyze_task_complexity` intelligently
- **State management**: All agents share and update the same state
- **Message history**: Context flows seamlessly between agents
- **Error handling**: Graceful fallbacks if anything goes wrong

### 🎯 The Result

You'll get a rich conversation state with:
- ✅ **Structured messages** from each agent
- ✅ **Detailed plan** created by the specialist
- ✅ **Full chat history** for continued conversation
- ✅ **Agent attribution** for debugging
- ✅ **Tool outputs** integrated seamlessly

In [ ]:
# 🚀 Create our multi-agent system
print("🏗️ Setting up the multi-agent system...")

# Initialize the conversation state
conversation = ConversationState()

# Add a complex user request
user_request = """
I want to organize a company retreat for 50 people that includes team building activities, 
educational workshops, accommodation, catering, and transportation. The retreat should be 
3 days long and help improve team collaboration and skills development.
"""

conversation.add_message("user", user_request.strip())

print(f"👤 User request: {user_request.strip()[:100]}...")

# Create the orchestrator (which includes the planner)
orchestrator = OrchestratorAgent()

print("\n🎭 Running the orchestrator...")

# Run the orchestrator - watch the magic happen!
result = await orchestrator.run(conversation)

print("\n✨ Multi-agent orchestration complete!\n")

# Display the results
result

---

## 🔍 Step 6: Explore the Rich Results

Let's examine what our multi-agent system produced. You'll see how **each component** contributed to the final result:

### 📊 Conversation Messages
See how different agents contributed to the conversation:

In [ ]:
print("📝 CONVERSATION MESSAGES:")
print("=" * 50)

for i, message in enumerate(result.messages):
    role = message["role"].upper()
    agent = message.get("agent_name", "unknown")
    content = message["content"][:200] + "..." if len(message["content"]) > 200 else message["content"]
    
    print(f"\n{i+1}. [{role}] ({agent})")
    print(f"   {content}")

print(f"\n📊 Total messages: {len(result.messages)}")

### 📋 Generated Plan
See the detailed plan created by our specialized planning agent:

In [ ]:
print("📋 GENERATED PLAN:")
print("=" * 50)

if result.plan:
    for i, step in enumerate(result.plan):
        print(f"{i+1}. {step}")
    print(f"\n🎯 Total plan steps: {len(result.plan)}")
else:
    print("No plan was generated.")

print("\n📈 Plan Summary:")
print(result.get_plan_summary())

### 💬 Chat History
See how message history is maintained for LLM context:

In [ ]:
print("💬 CHAT HISTORY (for LLM context):")
print("=" * 50)

if result.chat_history:
    for i, msg in enumerate(result.chat_history[-5:]):  # Show last 5 messages
        print(f"{i+1}. {msg[:150]}..." if len(msg) > 150 else f"{i+1}. {msg}")
    
    if len(result.chat_history) > 5:
        print(f"\n... and {len(result.chat_history) - 5} more messages")
    
    print(f"\n📊 Total chat history entries: {len(result.chat_history)}")
else:
    print("No chat history available.")

---

## 🔄 Step 7: Continue the Conversation - Multi-Turn Magic

One of the most powerful features is **conversational continuity**. Let's see how our agents maintain context across multiple interactions:

### 🌟 Why This Works So Well

- **Persistent State**: All conversation history is preserved
- **Agent Memory**: Each agent remembers previous interactions
- **Context Aware**: New requests can reference previous plans
- **Seamless Flow**: No context loss between turns

In [ ]:
print("🔄 CONTINUING THE CONVERSATION...")
print("=" * 50)

# Add a follow-up question that references the previous plan
follow_up = "That's a great plan! Can you modify it to be more budget-friendly and focus on local venues?"

result.add_message("user", follow_up)

print(f"👤 Follow-up: {follow_up}")
print("\n🎭 Running orchestrator again with updated context...")

# Run the orchestrator again - it has full context!
updated_result = await orchestrator.run(result)

print("\n✨ Context-aware response generated!")

# Show the latest response
latest_response = updated_result.get_latest_assistant_message()
print(f"\n🤖 Latest response: {latest_response[:300]}..." if len(latest_response) > 300 else f"\n🤖 Latest response: {latest_response}")

print(f"\n📊 Conversation now has {len(updated_result.messages)} total messages")
print(f"📋 Plan now has {len(updated_result.plan)} steps")

---

## 🎯 Step 8: Demonstrate Agent Specialization

Let's show how different types of requests are handled by appropriate specialists:

In [ ]:
print("🎯 TESTING AGENT SPECIALIZATION")
print("=" * 50)

# Test 1: Direct question (should be handled by orchestrator directly)
print("\n🧪 Test 1: Simple question")
simple_state = ConversationState()
simple_state.add_message("user", "What's the weather like today?")

simple_result = await orchestrator.run(simple_state)
print(f"📝 Response: {simple_result.get_latest_assistant_message()[:150]}...")
print(f"🔧 Tools used: {'planner' if len(simple_result.plan) > 0 else 'none'}")

# Test 2: Planning request (should delegate to planner)
print("\n🧪 Test 2: Planning request")
planning_state = ConversationState()
planning_state.add_message("user", "Help me plan a study schedule for learning Python programming")

planning_result = await orchestrator.run(planning_state)
print(f"📝 Response: {planning_result.get_latest_assistant_message()[:150]}...")
print(f"🔧 Plan generated: {len(planning_result.plan)} steps")
print(f"📋 First plan step: {planning_result.plan[0] if planning_result.plan else 'None'}")

print("\n✅ Agent specialization working perfectly!")

---

## 🎓 What You've Built: A Production-Ready Architecture

Congratulations! You've just built a sophisticated multi-agent system that demonstrates Orqest's most powerful patterns:

### 🏗️ **Architecture Overview**

```
🎭 OrchestratorAgent (Master Controller)
│
├── 🎯 PlannerAgent (Planning Specialist)
│   └── 🔧 analyze_task_complexity (Tool)
│
├── 💬 ConversationState (Shared Memory)
│   ├── 📝 messages (Structured conversation)
│   ├── 📋 plan (Generated plans)
│   └── 💭 chat_history (LLM context)
│
└── 🔄 Message History (Context continuity)
```

### 🌟 **Key Patterns You've Mastered**

| Pattern | What You Built | Why It's Powerful |
|---------|----------------|-------------------|
| **Agent as Tools** | Orchestrator uses PlannerAgent as a tool | Reusable, specialized capabilities |
| **Shared State** | ConversationState flows between agents | Consistent data across all interactions |
| **Message History** | LLM context preserved across calls | Conversational continuity |
| **Tool Integration** | Planner uses complexity analysis tool | Enhanced agent capabilities |
| **Error Handling** | Graceful fallbacks at every level | Production-ready resilience |
| **Delegation** | Smart routing based on request type | Optimal resource utilization |

### 🚀 **Ready for Production**

Your system now has:
- ✅ **Modular Design**: Easy to add new agents and tools
- ✅ **Type Safety**: All interactions are strongly typed
- ✅ **Observability**: Comprehensive logging and debugging
- ✅ **Scalability**: Each agent can be optimized independently
- ✅ **Testability**: Mock any component for unit testing
- ✅ **Extensibility**: Add new capabilities without breaking existing code

### 🎯 **Next Steps: Take It Further**

Now that you understand the patterns, you can:

1. **Add More Specialists**:
   ```python
   class ExecutorAgent(BaseAgent[ConversationState]):
       # Execute plans created by PlannerAgent
   
   class AnalyzerAgent(BaseAgent[ConversationState]):
       # Analyze results and provide insights
   ```

2. **Create Complex Tools**:
   ```python
   def _search_knowledge_base(self, ctx: RunContext[ConversationState], query: str):
       # Search external knowledge sources
   
   def _validate_plan_feasibility(self, ctx: RunContext[ConversationState], plan: List[str]):
       # Check if a plan is realistic and achievable
   ```

3. **Build Workflow Chains**:
   ```python
   # Chain agents together for complex workflows
   plan_result = await planner_agent.run(state)
   execution_result = await executor_agent.run(plan_result)
   analysis_result = await analyzer_agent.run(execution_result)
   ```

4. **Add Advanced State Management**:
   ```python
   class EnhancedState(ConversationState):
       user_preferences: Dict[str, Any] = Field(default_factory=dict)
       task_history: List[Dict[str, Any]] = Field(default_factory=list)
       performance_metrics: Dict[str, float] = Field(default_factory=dict)
   ```

---

## 🎉 Congratulations!

You've mastered **Orqest's Agent-as-Tools pattern** and built a production-ready multi-agent system! 

**What makes this special:**
- 🎯 **Each agent excels at one thing** instead of trying to do everything
- 🔗 **Agents work together seamlessly** through the shared state pattern
- 💬 **Conversations flow naturally** with persistent message history
- 🛠️ **Easy to extend and modify** without breaking existing functionality
- 🔍 **Fully observable and debuggable** at every level

You're now ready to build **any complexity of AI system** using these modular, composable patterns. Welcome to the future of AI development! 🚀